# Notebook A (indoor-light variant): phone caries detector robust to warm, dim light

Generated by `photo_screening/scripts/build_light_notebook.py` from `train_caries_aku.ipynb`; identical except
for the cells marked **L1–L3**: one warm, dimmed copy of every training photo (labels unchanged), a fixed
warm-light copy of the test split, and a side-by-side evaluation against the current production model.
Validation (and therefore threshold calibration) uses the original photos only.

In [ ]:
# 1. Environment
import os, sys, time, json, shutil, subprocess, platform, random, hashlib, zipfile
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if Path("/content").exists() else Path.cwd()
DATA = Path("/tmp/aku") if ON_KAGGLE else WORK / "aku"      # keep the dataset out of the saved output
OUT = WORK / "caries_aku_artifacts"
for p in (DATA, OUT, OUT / "plots", OUT / "samples"):
    p.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3", "requests"], check=True)
import numpy as np, requests, torch, ultralytics
from ultralytics import YOLO

SEED = 0
random.seed(SEED); np.random.seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
log(f"python {platform.python_version()} | torch {torch.__version__} | ultralytics {ultralytics.__version__} | "
    f"device {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'} | work {WORK}")
if DEVICE == "cpu":
    log("WARNING: no GPU — enable a GPU accelerator, otherwise training will not finish in time.")

In [ ]:
# 2. Helpers: resumable download with size check, box metrics
def download(url, dest, expected_size=None, tries=5):
    dest = Path(dest)
    for attempt in range(1, tries + 1):
        have = dest.stat().st_size if dest.exists() else 0
        if expected_size and have == expected_size:
            return dest
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                if r.status_code == 416:                       # already complete
                    return dest
                r.raise_for_status()
                mode = "ab" if (have and r.status_code == 206) else "wb"
                total = int(r.headers.get("Content-Length", 0)) + (have if mode == "ab" else 0)
                done, last = (have if mode == "ab" else 0), time.time()
                with open(dest, mode) as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk); done += len(chunk)
                        if time.time() - last > 20:
                            log(f"  {dest.name}: {done / 1e9:.2f} / {total / 1e9:.2f} GB"); last = time.time()
        except Exception as e:                                     # network hiccup -> resume
            log(f"  download attempt {attempt} failed: {e}")
            time.sleep(5 * attempt)
            continue
        if not expected_size or dest.stat().st_size == expected_size:
            return dest
        log(f"  size mismatch ({dest.stat().st_size} != {expected_size}), retrying")
    raise RuntimeError(f"download failed: {url}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def iou_matrix(a, b):
    """IoU between xyxy arrays a (N,4) and b (M,4)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def match(preds, gts, iou_thr=0.5):
    """preds: {img: (boxes, scores)}, gts: {img: boxes}. Returns (scores, is_tp, n_gt), greedy by score."""
    scores, tps, n_gt = [], [], 0
    for img, g in gts.items():
        n_gt += len(g)
        b, s = preds.get(img, (np.zeros((0, 4)), np.zeros(0)))
        order = np.argsort(-s); b, s = b[order], s[order]
        used = np.zeros(len(g), bool); ious = iou_matrix(b, g)
        for i in range(len(b)):
            j = int(np.argmax(ious[i])) if len(g) else -1
            ok = j >= 0 and ious[i, j] >= iou_thr and not used[j]
            if ok:
                used[j] = True
            scores.append(s[i]); tps.append(ok)
    return np.array(scores), np.array(tps, bool), n_gt


def pr_at(scores, tps, n_gt, thr):
    keep = scores >= thr
    tp = int(tps[keep].sum()); fp = int(keep.sum() - tp)
    p = tp / max(tp + fp, 1); r = tp / max(n_gt, 1)
    return {"conf": float(thr), "precision": p, "recall": r, "f1": 2 * p * r / max(p + r, 1e-9), "tp": tp, "fp": fp, "fn": n_gt - tp}


def ap50(scores, tps, n_gt):
    """101-point interpolated AP (same interpolation Ultralytics uses)."""
    if n_gt == 0 or len(scores) == 0:
        return 0.0
    o = np.argsort(-scores); tp = np.cumsum(tps[o]); fp = np.cumsum(~tps[o])
    rec = tp / n_gt; prec = tp / np.maximum(tp + fp, 1e-9)
    mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[1.0], prec, [0.0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(np.trapezoid(np.interp(x, mrec, mpre), x) if hasattr(np, "trapezoid") else np.trapz(np.interp(x, mrec, mpre), x))


def image_level(preds, gts, thr):
    """'Does this image contain any caries?' sensitivity / specificity at a score threshold."""
    tp = fp = tn = fn = 0
    for img, g in gts.items():
        s = preds.get(img, (None, np.zeros(0)))[1]
        pos_pred, pos_true = bool((s >= thr).any()), len(g) > 0
        tp += pos_pred and pos_true; fn += (not pos_pred) and pos_true
        fp += pos_pred and not pos_true; tn += (not pos_pred) and not pos_true
    return {"conf": float(thr), "sensitivity": tp / max(tp + fn, 1), "specificity": tn / max(tn + fp, 1),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn, "n_images": len(gts)}

log("helpers ready")

In [ ]:
# 3. Download the official Benchmarking Dataset (sizes verified against the Zenodo API)
REC = requests.get("https://zenodo.org/api/records/14827784", timeout=60).json()
FILES = {f["key"]: f for f in REC["files"]}
log(f"Zenodo: {REC['metadata']['title']} | licence {REC['metadata'].get('license', {}).get('id')} | doi {REC.get('doi')}")
print(requests.get(FILES["Data tree.txt"]["links"]["self"], timeout=60).text[:1200])

zf = FILES["Benchmarking Dataset.zip"]
zpath = download(zf["links"]["self"], DATA / "benchmarking.zip", expected_size=zf["size"])
log(f"downloaded {zpath.stat().st_size / 1e9:.2f} GB; extracting")
with zipfile.ZipFile(zpath) as z:
    z.extractall(DATA / "raw")
zpath.unlink()                                                     # free disk on Kaggle
log("extracted")

In [ ]:
# 4. Inspect the extracted layout instead of assuming it, then build a flat YOLO view
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
raw = DATA / "raw"

def visible(p):                                           # skip macOS zip junk (__MACOSX/, ._AppleDouble files)
    return not p.name.startswith("._") and "__MACOSX" not in p.parts and p.name != ".DS_Store"
for p in sorted(raw.rglob("*")):
    if p.is_dir() and visible(p) and len(p.relative_to(raw).parts) <= 3:
        n = sum(1 for q in p.iterdir() if q.is_file() and visible(q))
        print(f"{'  ' * (len(p.relative_to(raw).parts) - 1)}{p.name}/  ({n} files)")

def split_dir(name):
    hits = [p for p in raw.rglob(name) if p.is_dir() and visible(p) and (p / "images").exists() and (p / "yolo").exists()]
    if not hits:
        raise SystemExit(f"Could not find a '{name}' split with images/ and yolo/ folders — inspect the tree above.")
    return hits[0]

SPLITS = {s: split_dir(s) for s in ("train", "valid", "test")}

# Class names: classes.txt / data.yaml inside yolo/, else COCO categories, else generic.
def class_names():
    y = SPLITS["train"] / "yolo"
    for cand in [c for c in list(y.rglob("classes.txt")) + list(y.rglob("obj.names")) if visible(c)]:
        names = [l.strip() for l in cand.read_text().splitlines() if l.strip()]
        if names:
            return names, str(cand.relative_to(raw))
    for cand in filter(visible, y.rglob("*.yaml")):
        import yaml
        d = yaml.safe_load(cand.read_text())
        if isinstance(d, dict) and d.get("names"):
            n = d["names"]
            return (list(n.values()) if isinstance(n, dict) else list(n)), str(cand.relative_to(raw))
    for cand in filter(visible, (SPLITS["train"] / "coco").rglob("*.json")):
        cats = json.loads(cand.read_text()).get("categories", [])
        if cats:
            return [c["name"] for c in sorted(cats, key=lambda c: c["id"])], str(cand.relative_to(raw))
    return None, None

NAMES, NAMES_SRC = class_names()
VIEW = DATA / "yolo_view"
stats = {}
max_cls = -1
for split, d in SPLITS.items():
    labels = {p.stem: p for p in (d / "yolo").rglob("*.txt") if visible(p) and p.name not in ("classes.txt",)}
    imgs = [p for p in (d / "images").rglob("*") if visible(p) and p.suffix.lower() in IMG_EXT]
    (VIEW / split / "images").mkdir(parents=True, exist_ok=True); (VIEW / split / "labels").mkdir(parents=True, exist_ok=True)
    n_lab = n_box = 0
    for p in imgs:
        dst = VIEW / split / "images" / p.name
        if not dst.exists():
            dst.symlink_to(p)
        lab = labels.get(p.stem)
        out = VIEW / split / "labels" / f"{p.stem}.txt"
        if lab is not None:
            rows = [l.split() for l in lab.read_text(errors="replace").splitlines() if l.strip()]
            rows = [r for r in rows if len(r) == 5]                 # boxes only
            for r in rows:
                max_cls = max(max_cls, int(float(r[0])))
            out.write_text("".join(" ".join(r) + "\n" for r in rows))
            n_lab += 1; n_box += len(rows)
        elif not out.exists():
            out.write_text("")                                     # background image
    stats[split] = {"images": len(imgs), "label_files": n_lab, "boxes": n_box}
    log(f"{split}: {len(imgs)} images, {n_lab} label files, {n_box} boxes")

if stats["train"]["boxes"] == 0:
    raise SystemExit("No YOLO box rows found in train/yolo — the label format differs from expected; inspect the tree above.")
if NAMES is None or len(NAMES) <= max_cls:
    log(f"class names unresolved (found {NAMES} from {NAMES_SRC}, max class id {max_cls}); using generic names")
    NAMES = [f"caries_{i}" for i in range(max_cls + 1)] if max_cls > 0 else ["caries"]
log(f"classes ({NAMES_SRC}): {NAMES}")

DATA_YAML = DATA / "data.yaml"
DATA_YAML.write_text(json.dumps({"path": str(VIEW), "train": "train/images", "val": "valid/images",
                                 "test": "test/images", "names": NAMES}))
test_files = sorted(p.name for p in (VIEW / "test" / "images").iterdir() if visible(p))
(OUT / "test_files.txt").write_text("\n".join(test_files) + "\n")
log(f"data.yaml written; test split = {len(test_files)} images (list saved)")

In [ ]:
# L1. Indoor-light augmentation: a warm, dimmed copy of every TRAIN image; a fixed warm copy of TEST
from PIL import Image

def indoor_light(path, rng):
    a = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255
    a = a * np.array([rng.uniform(1.05, 1.30), rng.uniform(0.95, 1.05), rng.uniform(0.65, 0.90)])  # warm cast
    a = np.clip(a, 0, 1) ** rng.uniform(1.2, 1.9)                                                    # dimmer
    return Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))

def light_copies(src_split, dst_split, seed, suffix):
    rng = np.random.default_rng(seed)
    (VIEW / dst_split / "images").mkdir(parents=True, exist_ok=True)
    (VIEW / dst_split / "labels").mkdir(parents=True, exist_ok=True)
    n = 0
    for img in sorted(filter(visible, (VIEW / src_split / "images").iterdir())):
        if img.stem.endswith("_warm"):
            continue
        out = VIEW / dst_split / "images" / f"{img.stem}{suffix}.jpg"
        indoor_light(img, rng).save(out, quality=90)
        lab = VIEW / src_split / "labels" / f"{img.stem}.txt"
        (VIEW / dst_split / "labels" / f"{img.stem}{suffix}.txt").write_text(lab.read_text() if lab.exists() else "")
        n += 1
    return n

n_train = light_copies("train", "train", seed=SEED, suffix="_warm")          # train set doubles
n_test = light_copies("test", "test_warm", seed=20261016, suffix="")         # same names, warm pixels
cfg = json.loads(DATA_YAML.read_text()); cfg["test_warm"] = "test_warm/images"
DATA_YAML.write_text(json.dumps(cfg))
log(f"indoor-light copies: train +{n_train}, test_warm {n_test}")

In [ ]:
# 5. Train YOLOv8s (hard time cap keeps the whole notebook inside ~3 h on a T4)
model = YOLO("yolov8s.pt")
results = model.train(data=str(DATA_YAML), imgsz=640, epochs=60, patience=15, batch=-1, seed=SEED,
                      deterministic=True, device=DEVICE, workers=4, time=2.5,
                      project=str(WORK / "runs"), name="caries_aku", exist_ok=True, plots=True, verbose=False)
RUN = Path(model.trainer.save_dir)
BEST = RUN / "weights" / "best.pt"
log(f"training done -> {BEST} ({BEST.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# 6. Collect raw predictions once per split (conf 0.001) for calibration and our own metrics
def load_gts(split):
    gts = {}
    for img in sorted(filter(visible, (VIEW / split / "images").iterdir())):
        lab = VIEW / split / "labels" / f"{img.stem}.txt"
        rows = [list(map(float, l.split())) for l in lab.read_text().splitlines() if l.strip()] if lab.exists() else []
        gts[img.name] = rows                                       # normalized cx, cy, w, h (cls dropped below)
    return gts

def to_xyxy(rows, w, h):
    if not rows:
        return np.zeros((0, 4))
    a = np.array([r[1:5] for r in rows])
    return np.stack([(a[:, 0] - a[:, 2] / 2) * w, (a[:, 1] - a[:, 3] / 2) * h,
                     (a[:, 0] + a[:, 2] / 2) * w, (a[:, 1] + a[:, 3] / 2) * h], 1)

def predict_split(weights, split, imgsz, keep_cls=None, batch=32):
    m = YOLO(str(weights))
    keep = set(keep_cls) if keep_cls is not None else None
    preds, gts = {}, {}
    raw_gts = load_gts(split)
    files = sorted(filter(visible, (VIEW / split / "images").iterdir()))
    for i in range(0, len(files), batch):
        for r in m.predict([str(f) for f in files[i:i + batch]], imgsz=imgsz, conf=0.001, iou=0.6,
                           device=DEVICE, verbose=False, max_det=100):
            name = Path(r.path).name; h, w = r.orig_shape
            b = r.boxes; cls = b.cls.cpu().numpy().astype(int)
            sel = np.ones(len(cls), bool) if keep is None else np.isin(cls, list(keep))
            preds[name] = (b.xyxy.cpu().numpy()[sel], b.conf.cpu().numpy()[sel])
            gts[name] = to_xyxy(raw_gts[name], w, h)               # all caries classes merged
    return preds, gts

val_preds, val_gts = predict_split(BEST, "valid", 640)
test_preds, test_gts = predict_split(BEST, "test", 640)
log(f"predictions cached: valid {len(val_preds)} imgs, test {len(test_preds)} imgs")

In [ ]:
# L2. Predictions on the warm test copy, for the new model and for the current production model
import glob
test_warm_preds, test_warm_gts = predict_split(BEST, "test_warm", 640)
PREV = next(iter(glob.glob("/kaggle/input/**/caries_aku_artifacts/best.pt", recursive=True)), None)
prev = {"status": "previous model not attached"}
if PREV:
    prev_th = json.loads((Path(PREV).parent / "thresholds.json").read_text())["classes"]["caries"]
    prev_test, _ = predict_split(PREV, "test", 640)
    prev_warm, _ = predict_split(PREV, "test_warm", 640)
    prev = {"status": "ok", "weights_sha256": sha256(Path(PREV)), "thresholds": prev_th}
log(f"warm-light predictions: new {len(test_warm_preds)} imgs; previous model: {prev['status']}")

In [ ]:
# 7. Calibrate operating thresholds on VALID (screening favours sensitivity). Ordered by construction:
#    Same rule as photo_screening/scripts/calibrate.py.
s, tp, n = match(val_preds, val_gts)
grid = np.round(np.arange(0.01, 0.96, 0.01), 2)
curve = [pr_at(s, tp, n, t) for t in grid]

def highest_conf_with_recall(target, default):
    ok = [c["conf"] for c in curve if c["recall"] >= target]
    return max(ok) if ok else default

t_low = highest_conf_with_recall(0.90, 0.05)          # keep: sensitivity-first
best_f1 = max(curve, key=lambda c: c['f1'])
t_med = max(t_low, best_f1['conf'])        # best balance, never below the keep threshold
prec90 = [c["conf"] for c in curve if c["precision"] >= 0.90 and c["conf"] >= t_med]
t_high = min(prec90) if prec90 else round(min(t_med + 0.10, 0.95), 2)
thresholds = {"model_id": "caries_aku", "classes": {"caries": {"t_low": t_low, "t_med": t_med, "t_high": t_high}},
              "method": "valid split, box-level IoU>=0.5; t_low = highest conf with recall>=0.90; "
                        "t_med = max-F1 conf (>= t_low); t_high = lowest conf >= t_med with precision>=0.90",
              "valid_curve": curve}
(OUT / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
log(f"thresholds: low {t_low}  med {t_med}  high {t_high}")

In [ ]:
# 8. Evaluate on the held-out TEST split — Ultralytics val() + our own matching
PUBLISHED_BASELINE_MAP50 = 0.841   # YOLOv8s on the same official test split, as published with the dataset
val = YOLO(str(BEST)).val(data=str(DATA_YAML), split="test", imgsz=640, batch=16, conf=0.001, iou=0.6,
                          device=DEVICE, plots=True, project=str(WORK / "runs"), name="caries_aku_test",
                          exist_ok=True, verbose=False)
per_class = {NAMES[int(c)]: {"ap50": float(val.box.ap50[i]), "ap50_95": float(val.box.ap[i])}
             for i, c in enumerate(val.box.ap_class_index)}
s, tp, n = match(test_preds, test_gts)
test_metrics = {
    "split": "test (official Benchmarking Dataset split)",
    "n_images": len(test_gts), "n_boxes": int(n),
    "ultralytics": {"precision": float(val.box.mp), "recall": float(val.box.mr),
                    "map50": float(val.box.map50), "map50_95": float(val.box.map), "per_class": per_class},
    "merged_caries": {"ap50": ap50(s, tp, n),
                      "at_t_low": pr_at(s, tp, n, t_low), "at_t_med": pr_at(s, tp, n, t_med),
                      "at_0.25": pr_at(s, tp, n, 0.25)},
    "image_level": {"at_t_low": image_level(test_preds, test_gts, t_low),
                    "at_t_med": image_level(test_preds, test_gts, t_med)},
    "published_baseline": {"model": "YOLOv8s", "map50": PUBLISHED_BASELINE_MAP50,
                           "delta_map50": float(val.box.map50) - PUBLISHED_BASELINE_MAP50},
}
for f in ("PR_curve.png", "BoxPR_curve.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "F1_curve.png", "BoxF1_curve.png"):
    p = Path(val.save_dir) / f
    if p.exists():
        shutil.copy(p, OUT / "plots" / f"test_{f}")
for f in ("results.png", "results.csv", "args.yaml"):
    if (RUN / f).exists():
        shutil.copy(RUN / f, OUT / "plots" / f if f.endswith(".png") else OUT / f)
log(f"TEST mAP@50 {val.box.map50:.3f} (published YOLOv8s baseline {PUBLISHED_BASELINE_MAP50}) | "
    f"P {val.box.mp:.3f} R {val.box.mr:.3f} | image-level sens {test_metrics['image_level']['at_t_low']['sensitivity']:.3f} "
    f"spec {test_metrics['image_level']['at_t_low']['specificity']:.3f} @ t_low")

In [ ]:
# L3. Normal vs warm light, new vs previous model (each at its own calibrated keep threshold)
def summary(preds, gts, t):
    s, tp, n = match(preds, gts)
    return {"ap50": ap50(s, tp, n), "image_level": image_level(preds, gts, t)}

indoor = {"new_model": {"test": summary(test_preds, test_gts, t_low),
                        "test_warm": summary(test_warm_preds, test_warm_gts, t_low), "t_low": t_low}}
if prev["status"] == "ok":
    indoor["previous_model"] = {"test": summary(prev_test, test_gts, prev_th["t_low"]),
                                "test_warm": summary(prev_warm, test_warm_gts, prev_th["t_low"]),
                                "t_low": prev_th["t_low"], "weights_sha256": prev["weights_sha256"]}
test_metrics["indoor_light"] = indoor
for k, v in indoor.items():
    log(f"{k}: test AP50 {v['test']['ap50']:.3f} sens {v['test']['image_level']['sensitivity']:.3f} "
        f"spec {v['test']['image_level']['specificity']:.3f} | warm AP50 {v['test_warm']['ap50']:.3f} "
        f"sens {v['test_warm']['image_level']['sensitivity']:.3f} "
        f"spec {v['test_warm']['image_level']['specificity']:.3f}")

In [ ]:
# 9. Cross-domain: AlphaDent (DSLR/macro-trained) caries class on the same phone test split
import gc
cross = {"status": "skipped"}
for _name in ("model", "results", "val"):
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache() if DEVICE == 0 else None          # training left VRAM reserved
try:
    rel = requests.get("https://api.github.com/repos/ZFTurbo/AlphaDent/releases/tags/v1.0", timeout=60).json()
    asset = next(a for a in rel["assets"] if a["name"] == "yolov8x_AlphaDent_4_classes_960px.pt")
    ad_path = download(asset["browser_download_url"], DATA / asset["name"], expected_size=asset["size"])
    ad_names = YOLO(str(ad_path)).names
    caries_ids = [i for i, n in ad_names.items() if "caries" in str(n).lower()]
    log(f"AlphaDent classes {ad_names}; caries ids {caries_ids}")
    ad_preds, ad_gts = predict_split(ad_path, "test", 960, keep_cls=caries_ids, batch=4)
    s, tp, n = match(ad_preds, ad_gts)
    cross = {"status": "ok", "model": asset["name"], "sha256": sha256(ad_path), "classes": {int(k): v for k, v in ad_names.items()},
             "caries_class_ids": caries_ids, "imgsz": 960,
             "merged_caries": {"ap50": ap50(s, tp, n), "at_0.25": pr_at(s, tp, n, 0.25), "at_0.10": pr_at(s, tp, n, 0.10)},
             "image_level": {"at_0.25": image_level(ad_preds, ad_gts, 0.25), "at_0.10": image_level(ad_preds, ad_gts, 0.10)},
             "note": "AlphaDent caries labels follow its own annotation protocol (DSLR/macro photos); "
                     "differences from AKU labelling contribute to the gap alongside the camera domain."}
    log(f"AlphaDent caries on AKU test: AP50 {cross['merged_caries']['ap50']:.3f} vs AKU YOLOv8s "
        f"{test_metrics['merged_caries']['ap50']:.3f}")
except Exception as e:                                             # comparison is optional; never lose the main result
    cross = {"status": f"failed: {e}"}
    log(f"cross-domain comparison failed: {e}")

In [ ]:
# 10. Samples for the app's "Try sample photos" button (CC BY 4.0, attributed)
rng = random.Random(SEED)
pos = [k for k, g in test_gts.items() if len(g)]; neg = [k for k, g in test_gts.items() if not len(g)]
picks = rng.sample(pos, min(3, len(pos))) + rng.sample(neg, min(2, len(neg)))
rest = [k for k in test_gts if k not in picks]
picks += rng.sample(rest, min(5 - len(picks), len(rest)))
for i, name in enumerate(picks, 1):
    src = (VIEW / "test" / "images" / name).resolve()
    shutil.copy(src, OUT / "samples" / f"aku_sample_{i}{src.suffix.lower()}")
(OUT / "samples" / "ATTRIBUTION.md").write_text(
    "Sample images from the AKU intraoral caries dataset (Benchmarking Dataset, test split).\n\n"
    f"Source: {REC['metadata']['title']}, Zenodo, doi:{REC.get('doi')}. Licence: CC BY 4.0 "
    "(https://creativecommons.org/licenses/by/4.0/). Images unmodified; renamed.\n\n"
    + "\n".join(f"- aku_sample_{i}: test/{n}" for i, n in enumerate(picks, 1)) + "\n")
log(f"{len(picks)} samples copied")

In [ ]:
# 11. Package
shutil.copy(BEST, OUT / "best.pt")
metrics = {
    "model_id": "caries_aku", "arch": "yolov8s", "imgsz": 640, "seed": SEED,
    "best_pt_sha256": sha256(OUT / "best.pt"), "classes": NAMES, "class_names_source": NAMES_SRC,
    "dataset": {"name": REC["metadata"]["title"], "doi": REC.get("doi"), "licence": "CC-BY-4.0",
                "zip": "Benchmarking Dataset.zip", "splits": stats},
    "train": {"run_dir": str(RUN), "results_csv": "results.csv"},
    "test": test_metrics, "cross_domain_alphadent": cross, "thresholds": {k: v for k, v in thresholds.items() if k != "valid_curve"},
    "env": {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
            "gpu": torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu", "kaggle": ON_KAGGLE},
    "wall_minutes": round((time.time() - T0) / 60, 1),
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
zip_path = shutil.make_archive(str(WORK / "caries_aku_artifacts"), "zip", OUT)
log(f"DONE -> {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
print(json.dumps({k: metrics[k] for k in ("test", "thresholds")}, indent=2)[:4000])